In [6]:
import numpy as np

from itertools import product

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from sklearn.ensemble import RandomForestClassifier 
# from sklearn.metrics import ( accuracy_score, classification_report ) 

In [7]:
# ============================================================
# 1. CREATE SAMPLE DATA
# ============================================================

X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    n_classes=2,
    random_state=42
)

print("Dataset shape:", X.shape)
print("Target shape:", y.shape)


# ============================================================
# 2. SPLIT DATA
# ============================================================

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", X_train.shape[0])
print("Validation samples:", X_val.shape[0])


# ============================================================
# 3. FEATURE SCALING
# ============================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)


Dataset shape: (1000, 10)
Target shape: (1000,)

Training samples: 800
Validation samples: 200


In [25]:
# 4. CREATE BASE MODEL # ============================================================ 
from sklearn.neural_network import MLPClassifier 

model = MLPClassifier( random_state=42, max_iter=500 )

# 5. DEFINE PARAMETER GRID 
# ============================================================ 
param_grid = { 
    "hidden_layer_sizes": [ (50,), (100,), (50, 50), (100, 50) ], 
    "activation": [ "relu", "tanh" ], 
    "solver": [ "adam" ], 
    "alpha": [ 0.0001, 0.001, 0.01 ], 
    "learning_rate": [ "constant", "adaptive" ] 
} 
print("\nNumber of parameter combinations:") 
total_combinations = ( len(param_grid["hidden_layer_sizes"]) * len(param_grid["activation"]) 
                       * len(param_grid["solver"]) * len(param_grid["alpha"]) 
                       * len(param_grid["learning_rate"]) ) 
print(total_combinations)


Number of parameter combinations:
48


In [26]:
from sklearn.model_selection import ( train_test_split, GridSearchCV, RandomizedSearchCV)

# 6. GRID SEARCH CV # ============================================================ 
grid_search = GridSearchCV( estimator=model, param_grid=param_grid, cv=5, scoring="accuracy", n_jobs=-1, verbose=1 ) 
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 48 candidates, totalling 240 fits


C:\Users\Shekhar Kumar\AppData\Roaming\Python\Python313\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",MLPClassifier...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'activation': ['relu', 'tanh'], 'alpha': [0.0001, 0.001, ...], 'hidden_layer_sizes': [(50,), (100,), ...], 'learning_rate': ['constant', 'adaptive'], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metriceva

In [27]:
# 7. GRID SEARCH RESULTS # ============================================================ 
print("\n================ GRID SEARCH RESULTS ================") 
print("Best Parameters:") 
print(grid_search.best_params_) 
print("\nBest CV Score:") 
print(grid_search.best_score_)


================ GRID SEARCH RESULTS ================
Best Parameters:
{'activation': 'tanh', 'alpha': 0.0001, 'hidden_layer_sizes': (50, 50), 'learning_rate': 'constant', 'solver': 'adam'}

Best CV Score:
0.91875


In [28]:
# 8. EVALUATE GRID SEARCH MODEL ON VALIDATION DATA 
# ============================================================


best_grid_model = grid_search.best_estimator_ 
y_pred_grid = best_grid_model.predict(X_val) 
grid_val_accuracy = accuracy_score( y_val, y_pred_grid ) 
print("\nValidation Accuracy:") 
print(grid_val_accuracy) 
print("\nClassification Report:") 
print(classification_report(y_val, y_pred_grid))


Validation Accuracy:
0.92

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.93      0.92       100
           1       0.93      0.91      0.92       100

    accuracy                           0.92       200
   macro avg       0.92      0.92      0.92       200
weighted avg       0.92      0.92      0.92       200



In [29]:
# ============================================================ 
# 9. RANDOMIZED SEARCH CV 
# ============================================================ 

random_search = RandomizedSearchCV( estimator=model, param_distributions=param_grid, 
                                    n_iter=20, cv=5, scoring="accuracy", random_state=42, n_jobs=-1, verbose=1 ) 
random_search.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


C:\Users\Shekhar Kumar\AppData\Roaming\Python\Python313\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",MLPClassifier...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'activation': ['relu', 'tanh'], 'alpha': [0.0001, 0.001, ...], 'hidden_layer_sizes': [(50,), (100,), ...], 'learning_rate': ['constant', 'adaptive'], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be s

In [30]:
# ============================================================ 
# 10. RANDOMIZED SEARCH RESULTS 
# ============================================================ 

print("\n================ RANDOMIZED SEARCH RESULTS ================") 
print("Best Parameters:") 
print(random_search.best_params_) 
print("\nBest CV Score:") 
print(random_search.best_score_)


================ RANDOMIZED SEARCH RESULTS ================
Best Parameters:
{'solver': 'adam', 'learning_rate': 'adaptive', 'hidden_layer_sizes': (50, 50), 'alpha': 0.001, 'activation': 'tanh'}

Best CV Score:
0.91875


In [31]:
# ============================================================ 
# 11. EVALUATE RANDOMIZED SEARCH MODEL 
# ============================================================ 
best_random_model = random_search.best_estimator_ 
y_pred_random = best_random_model.predict(X_val) 
random_val_accuracy = accuracy_score( y_val, y_pred_random ) 
print("\nValidation Accuracy:") 
print(random_val_accuracy) 
print("\nClassification Report:") 
print(classification_report(y_val, y_pred_random))


Validation Accuracy:
0.92

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.93      0.92       100
           1       0.93      0.91      0.92       100

    accuracy                           0.92       200
   macro avg       0.92      0.92      0.92       200
weighted avg       0.92      0.92      0.92       200



In [32]:
# ============================================================ 
# 12. FINAL COMPARISON 
# ============================================================ 
print("\n================ FINAL COMPARISON ================") 
print("GridSearchCV") 
print("Best Parameters:", grid_search.best_params_) 

print("Best CV Score:", grid_search.best_score_) 
print("Validation Accuracy:", grid_val_accuracy) 

print("\nRandomizedSearchCV") 
print("Best Parameters:", random_search.best_params_) 
print("Best CV Score:", random_search.best_score_) 
print("Validation Accuracy:", random_val_accuracy)


================ FINAL COMPARISON ================
GridSearchCV
Best Parameters: {'activation': 'tanh', 'alpha': 0.0001, 'hidden_layer_sizes': (50, 50), 'learning_rate': 'constant', 'solver': 'adam'}
Best CV Score: 0.91875
Validation Accuracy: 0.92

RandomizedSearchCV
Best Parameters: {'solver': 'adam', 'learning_rate': 'adaptive', 'hidden_layer_sizes': (50, 50), 'alpha': 0.001, 'activation': 'tanh'}
Best CV Score: 0.91875
Validation Accuracy: 0.92
